# FoodGuard AI - Data Preparation

This notebook:
1. **E-Nose Data**
2. **E-Tongue Data**
3. **Vision Deposit Patterns**
4. **Demo Samples**

---

## 1. Imports & Configuration

## 0. Populate Initial Batches from Sources


In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import json
from PIL import Image, ImageDraw
import random
from tqdm import tqdm

sys.path.insert(0, '..')

from foodguard_lib import (
    DB_PATH, SOURCES, DESCRIPTIONS,
    insert_batch, insert_aroma_analysis, insert_taste_analysis, insert_vision_analysis,
    execute_query, execute_insert, generate_batch_id, generate_vision_images,
    generate_enose_data, generate_etongue_data, populate_initial_batches, init_db
)

# Initialize database schema
print("Initializing database...")
init_db(DB_PATH)

# Populate initial batches from SOURCES and DESCRIPTIONS
print(f"\nSOURCES: {SOURCES}")
print(f"DESCRIPTIONS: {DESCRIPTIONS}")

batch_ids = populate_initial_batches(DB_PATH)
print(f"\n[OK] Created {len(batch_ids)} initial batches")
print(f"Batch IDs: {batch_ids[:5]}...")  # Show first 5

np.random.seed(42)
random.seed(42)

# Mock prediction function
def mock_predict_class(sample_type):
    """Mock ML classifier - returns predicted class and confidence."""
    return sample_type, 0.95

In [ ]:
# Generate and Insert E-Nose data for each batch
print("Generating and inserting E-Nose synthetic data...")
aroma_count = 0

# Generate E-Nose data (this creates all samples)
enose_df = generate_enose_data(n_samples_per_class=50)  # 50 samples per adulterant class
print(f"[OK] Generated {len(enose_df)} E-Nose samples")

# Distribute samples across batches
samples_per_batch = len(enose_df) // len(batch_ids)
sample_idx = 0

for batch_idx, batch_id in enumerate(batch_ids):
    batch_start = batch_idx * samples_per_batch
    batch_end = batch_start + samples_per_batch if batch_idx < len(batch_ids) - 1 else len(enose_df)

    for idx in range(batch_start, batch_end):
        if idx >= len(enose_df):
            break
        row = enose_df.iloc[idx]
        sample_type = row["adulterant"]
        pred_class, confidence = mock_predict_class(sample_type)

        try:
            insert_aroma_analysis(
                batch_id=batch_id,
                ammonia=float(row["ammonia"]),
                alcohol=float(row["alcohol"]),
                voc=float(row["voc"]),
                sulfur=float(row["sulfur"]),
                hydrocarbon=float(row["hydrocarbon"]),
                predicted_class=pred_class,
                confidence=confidence,
                db_path=DB_PATH
            )
            aroma_count += 1
        except Exception as e:
            print(f"[ERROR] Sample {idx}: {e}")
            break

        if (aroma_count) % 500 == 0:
            print(f"  ...inserted {aroma_count} E-Nose samples")

print(f"[OK] Inserted {aroma_count} E-Nose samples into DB")

Generating E-Nose synthetic data...
[OK] Generated 7000 E-Nose samples

E-Nose data sample:
  adulterant   ammonia   alcohol       voc    sulfur  hydrocarbon
0      Fresh  0.205341  0.312731  0.351593  0.252281     0.081248
1      Fresh  0.152355  0.225792  0.531760  0.192262     0.246061
2      Fresh  0.257249  0.544404  0.662782  0.079745     0.102513
3      Fresh  0.176966  0.549247  0.587401  0.052861     0.450729
4      Fresh  0.335817  0.382531  0.894869  0.302513     0.093792
5      Fresh  0.145208  0.616442  0.572782  0.099784     0.508891
6      Fresh  0.400222  0.232113  0.727373  0.093381     0.314477
7      Fresh  0.319095  0.670058  0.723399  0.277582     0.363807
8      Fresh  0.293148  0.261083  0.233732  0.203205     0.527062
9      Fresh  0.342418  0.324549  0.371121  0.294961     0.593640

E-Nose by class:
adulterant
Fresh        1000
Water        1000
Urea         1000
Detergent    1000
Formalin     1000
H2O2         1000
Spoiled      1000
Name: count, dtype: int64


## 3. E-Tongue Synthetic Data Generation

**Features**: sweetness, saltiness, sourness, bitterness, umami, astringency (6 sensors)

In [ ]:
# Generate and Insert E-Tongue data for each batch
print("Generating and inserting E-Tongue synthetic data...")
taste_count = 0

# Generate E-Tongue data (this creates all samples)
etongue_df = generate_etongue_data(n_samples_per_class=50)  # 50 samples per adulterant class
print(f"[OK] Generated {len(etongue_df)} E-Tongue samples")

# Distribute samples across batches
samples_per_batch = len(etongue_df) // len(batch_ids)

for batch_idx, batch_id in enumerate(batch_ids):
    batch_start = batch_idx * samples_per_batch
    batch_end = batch_start + samples_per_batch if batch_idx < len(batch_ids) - 1 else len(etongue_df)

    for idx in range(batch_start, batch_end):
        if idx >= len(etongue_df):
            break
        row = etongue_df.iloc[idx]
        sample_type = row["adulterant"]
        pred_class, confidence = mock_predict_class(sample_type)

        try:
            insert_taste_analysis(
                batch_id=batch_id,
                sweetness=float(row["sweetness"]),
                saltiness=float(row["saltiness"]),
                sourness=float(row["sourness"]),
                bitterness=float(row["bitterness"]),
                umami=float(row["umami"]),
                astringency=float(row["astringency"]),
                predicted_class=pred_class,
                confidence=confidence,
                db_path=DB_PATH
            )
            taste_count += 1
        except Exception as e:
            print(f"[ERROR] Sample {idx}: {e}")
            break

        if (taste_count) % 500 == 0:
            print(f"  ...inserted {taste_count} E-Tongue samples")

print(f"[OK] Inserted {taste_count} E-Tongue samples into DB")

Generating E-Tongue synthetic data...
[OK] Generated 7000 E-Tongue samples

E-Tongue data sample:
  adulterant  sweetness  saltiness  sourness  bitterness     umami  \
0      Fresh   5.239756   2.372301  0.706173    0.121964  0.764637   
1      Fresh   6.006347   1.418688  1.093750    0.342471  0.521868   
2      Fresh   6.125236   1.272289  0.465732    0.214309  0.683349   
3      Fresh   5.756209   2.294604  1.457749    0.259801  0.540307   
4      Fresh   5.336619   2.387838  0.872347    0.408727  0.704226   
5      Fresh   4.921458   1.416607  1.356537    0.120584  1.111064   
6      Fresh   3.739093   1.997609  1.123393    0.460281  0.614048   
7      Fresh   3.967556   1.296639  0.877499    0.196375  0.908602   
8      Fresh   5.636474   1.150595  1.048236    0.236977  0.956796   
9      Fresh   4.803759   1.699637  1.100362    0.129279  0.604843   

   astringency  
0     0.292168  
1     0.445908  
2     0.588943  
3     0.582916  
4     0.470816  
5     0.549719  
6     0.2008

In [ ]:
# Generate vision images
print("Generating vision deposit pattern images...")
vision_results = generate_vision_images(n_samples_per_class=30)  # 30 samples per adulterant class
print(f"[OK] Generated {len(vision_results)} vision images")

# Create mapping of images to batches
samples_per_batch = len(vision_results) // len(batch_ids)
print(f"Distribution: {samples_per_batch} images per batch")
print(f"\nSample vision results (first 5):")
for img_path, deposit_type, sample_type in vision_results[:5]:
    print(f"  {sample_type}: {img_path} [{deposit_type}]")

Generating vision deposit pattern images...
[OK] Generated 420 vision images

Sample vision results (first 5):
  Fresh: ../data/synthetic/vision/Fresh/fresh_0000.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0001.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0002.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0003.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0004.png [clean_ring]


In [ ]:
# Note: E-Nose data insertion is now done in the generation cell above
# This cell is kept for reference and can be skipped

print("[INFO] E-Nose data has already been inserted during generation")

Inserting E-Nose data into DB...
  ...inserted 1000/7000 E-Nose samples
  ...inserted 2000/7000 E-Nose samples
  ...inserted 3000/7000 E-Nose samples
  ...inserted 4000/7000 E-Nose samples
  ...inserted 5000/7000 E-Nose samples
  ...inserted 6000/7000 E-Nose samples
  ...inserted 7000/7000 E-Nose samples
[OK] Inserted 7000 E-Nose samples into DB


In [ ]:
# Note: E-Tongue data insertion is now done in the generation cell above
# This cell is kept for reference and can be skipped

print("[INFO] E-Tongue data has already been inserted during generation")

Inserting E-Tongue data into DB...
  ...inserted 1000/7000 E-Tongue samples
  ...inserted 2000/7000 E-Tongue samples
  ...inserted 3000/7000 E-Tongue samples
  ...inserted 4000/7000 E-Tongue samples
  ...inserted 5000/7000 E-Tongue samples
  ...inserted 6000/7000 E-Tongue samples
  ...inserted 7000/7000 E-Tongue samples
[OK] Inserted 7000 E-Tongue samples into DB


In [ ]:
# Insert Vision data into DB (distributed across batches)
print("Inserting Vision data into DB...")
vision_count = 0
samples_per_batch = len(vision_results) // len(batch_ids)

for batch_idx, batch_id in enumerate(batch_ids):
    batch_start = batch_idx * samples_per_batch
    batch_end = batch_start + samples_per_batch if batch_idx < len(batch_ids) - 1 else len(vision_results)

    for vis_idx in range(batch_start, batch_end):
        if vis_idx >= len(vision_results):
            break

        img_path, deposit_type, sample_type = vision_results[vis_idx]
        pred_class, confidence = mock_predict_class(sample_type)

        # Generate sample_id from image filename (remove extension)
        sample_id = Path(img_path).stem

        # Mock object detection info as JSON string
        # In a real scenario, this would come from the YOLO object detector
        object_detection_info = json.dumps({
            "total_objects_detected": 1,
            "object_class_counts": {deposit_type: 1},
            "detailed_instances": [{
                "object_class": deposit_type,
                "confidence": confidence,
                "bounding_box_xyxy": [10.0, 10.0, 250.0, 250.0]
            }]
        })

        try:
            insert_vision_analysis(
                sample_id=sample_id,
                batch_id=batch_id,
                image_path=img_path,
                deposit_type=deposit_type,
                predicted_class=pred_class,
                confidence=str(confidence),  # Convert to string
                object_detection_info=object_detection_info,
                db_path=DB_PATH
            )
            vision_count += 1
        except Exception as e:
            print(f"[ERROR] {img_path}: {e}")
            break

        if (vision_count) % 100 == 0:
            print(f"  ...inserted {vision_count}/{len(vision_results)} Vision samples")

print(f"[OK] Inserted {vision_count} Vision samples into DB")

Inserting Vision data into DB...
[ERROR] ../data/synthetic/vision/Fresh/fresh_0000.png: insert_vision_analysis() missing 2 required positional arguments: 'sample_id' and 'object_detection_info'
[OK] Inserted 0 Vision samples into DB


## 7. Verify Data in DB

In [18]:
# Check aroma_analysis table
aroma_query = "SELECT COUNT(*) as count FROM aroma_analysis"
aroma_rows = execute_query(DB_PATH, aroma_query)
print(f"Total aroma_analysis records: {dict(aroma_rows[0])['count']}")

# Check taste_analysis table
taste_query = "SELECT COUNT(*) as count FROM taste_analysis"
taste_rows = execute_query(DB_PATH, taste_query)
print(f"Total taste_analysis records: {dict(taste_rows[0])['count']}")

# Check vision_analysis table
vision_query = "SELECT COUNT(*) as count FROM vision_analysis"
vision_rows = execute_query(DB_PATH, vision_query)
print(f"Total vision_analysis records: {dict(vision_rows[0])['count']}")

Total aroma_analysis records: 14000
Total taste_analysis records: 14000
Total vision_analysis records: 0


## 8. Sample Data Inspection

In [19]:
# Show sample aroma record
query = "SELECT * FROM aroma_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Aroma Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

print("\n" + "="*50 + "\n")

# Show sample taste record
query = "SELECT * FROM taste_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Taste Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

print("\n" + "="*50 + "\n")

# Show sample vision record
query = "SELECT * FROM vision_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Vision Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

Sample Aroma Record:
  batch_id: BATCH-64AFD60E
  ground_truth: None
  ammonia: 0.2053408428170682
  alcohol: 0.31273084328308465
  voc: 0.3515926448329279
  sulfur: 0.2522810265691745
  hydrocarbon: 0.08124834044672548
  predicted_class: Fresh
  confidence: 0.95
  created_at: 2026-06-15 17:34:58


Sample Taste Record:
  batch_id: BATCH-DE1CF1FE
  ground_truth: None
  sweetness: 5.239755821423183
  saltiness: 2.372300670166239
  sourness: 0.7061726107133135
  bitterness: 0.12196356462089898
  umami: 0.7646366317697724
  astringency: 0.29216758197571485
  predicted_class: Fresh
  confidence: 0.95
  created_at: 2026-06-15 17:37:58




## Sample Data Generation for Demo

In [ ]:
import uuid, json
from pathlib import Path

# ── Helper ──────────────────────────────────────────────────────────────────
def make_sample_id(prefix, n):
    return f"{prefix}-S{n:03d}"

# ── Demo batch definitions (each batch = 1 Good + 1 Average + 1 Worst sample) ──
DEMO_BATCHES = [
    {
        "id":          "BATCH-DEMO-A",
        "source":      "Daily Cooperative Center",
        "description": "Mixed-quality run from cooperative centre – morning collection",
        "samples": [
            # ── GOOD: pure fresh milk ─────────────────────────────────────────
            {
                "sample_id":          "S001",
                "quality_tier":       "Good",
                "ground_truth":       "Fresh",
                "ammonia_signal":     0.05,
                "alcohol_signal":     0.04,
                "voc_signal":         0.06,
                "sulfur_signal":      0.02,
                "hydrocarbon_signal": 0.01,
                "sweetness":          5.8,
                "saltiness":          1.5,
                "sourness":           0.4,
                "bitterness":         0.10,
                "umami":              0.30,
                "astringency":        0.05,
                "globule_density":    0.90,
                "crystal_presence":   0.05,
                "deposit_type":       "clean_ring",
                "predicted_class":    "Fresh",
                "confidence":         0.97,
            },
            # ── AVERAGE: mild water adulteration ─────────────────────────────
            {
                "sample_id":          "S002",
                "quality_tier":       "Average",
                "ground_truth":       "Water",
                "ammonia_signal":     0.20,
                "alcohol_signal":     0.18,
                "voc_signal":         0.30,
                "sulfur_signal":      0.12,
                "hydrocarbon_signal": 0.08,
                "sweetness":          3.8,
                "saltiness":          1.9,
                "sourness":           1.1,
                "bitterness":         0.35,
                "umami":              0.20,
                "astringency":        0.18,
                "globule_density":    0.60,
                "crystal_presence":   0.20,
                "deposit_type":       "sparse_crystals",
                "predicted_class":    "Water",
                "confidence":         0.82,
            },
            # ── WORST: heavy urea contamination ──────────────────────────────
            {
                "sample_id":          "S003",
                "quality_tier":       "Worst",
                "ground_truth":       "Urea",
                "ammonia_signal":     0.85,
                "alcohol_signal":     0.70,
                "voc_signal":         0.90,
                "sulfur_signal":      0.75,
                "hydrocarbon_signal": 0.60,
                "sweetness":          1.5,
                "saltiness":          5.5,
                "sourness":           3.8,
                "bitterness":         2.50,
                "umami":              0.05,
                "astringency":        1.80,
                "globule_density":    0.15,
                "crystal_presence":   0.90,
                "deposit_type":       "crystalline_heavy",
                "predicted_class":    "Urea",
                "confidence":         0.98,
            },
        ],
    },
    {
        "id":          "BATCH-DEMO-B",
        "source":      "Village Milk Collection Unit",
        "description": "Mixed-quality run from village unit – evening collection",
        "samples": [
            # ── GOOD: excellent quality, slight natural variation ──────────────
            {
                "sample_id":          "S001",
                "quality_tier":       "Good",
                "ground_truth":       "Fresh",
                "ammonia_signal":     0.06,
                "alcohol_signal":     0.05,
                "voc_signal":         0.07,
                "sulfur_signal":      0.03,
                "hydrocarbon_signal": 0.02,
                "sweetness":          5.5,
                "saltiness":          1.6,
                "sourness":           0.5,
                "bitterness":         0.12,
                "umami":              0.28,
                "astringency":        0.06,
                "globule_density":    0.88,
                "crystal_presence":   0.06,
                "deposit_type":       "clean_ring",
                "predicted_class":    "Fresh",
                "confidence":         0.96,
            },
            # ── AVERAGE: borderline detergent trace ───────────────────────────
            {
                "sample_id":          "S002",
                "quality_tier":       "Average",
                "ground_truth":       "Detergent",
                "ammonia_signal":     0.35,
                "alcohol_signal":     0.25,
                "voc_signal":         0.45,
                "sulfur_signal":      0.20,
                "hydrocarbon_signal": 0.12,
                "sweetness":          3.2,
                "saltiness":          2.5,
                "sourness":           1.4,
                "bitterness":         0.60,
                "umami":              0.18,
                "astringency":        0.30,
                "globule_density":    0.55,
                "crystal_presence":   0.28,
                "deposit_type":       "sparse_crystals",
                "predicted_class":    "Detergent",
                "confidence":         0.78,
            },
            # ── WORST: antibiotic contamination ───────────────────────────────
            {
                "sample_id":          "S003",
                "quality_tier":       "Worst",
                "ground_truth":       "Antibiotic",
                "ammonia_signal":     0.78,
                "alcohol_signal":     0.65,
                "voc_signal":         0.82,
                "sulfur_signal":      0.68,
                "hydrocarbon_signal": 0.55,
                "sweetness":          1.8,
                "saltiness":          5.0,
                "sourness":           3.5,
                "bitterness":         3.00,
                "umami":              0.08,
                "astringency":        2.00,
                "globule_density":    0.12,
                "crystal_presence":   0.88,
                "deposit_type":       "crystalline_heavy",
                "predicted_class":    "Antibiotic",
                "confidence":         0.97,
            },
        ],
    },
    {
        "id":          "BATCH-DEMO-C",
        "source":      "Automated Milk Testing Lab",
        "description": "Mixed-quality run from automated lab – spot-check batch",
        "samples": [
            # ── GOOD: trace water dilution, still acceptable ───────────────────
            {
                "sample_id":          "S001",
                "quality_tier":       "Good",
                "ground_truth":       "Fresh",
                "ammonia_signal":     0.08,
                "alcohol_signal":     0.07,
                "voc_signal":         0.09,
                "sulfur_signal":      0.04,
                "hydrocarbon_signal": 0.03,
                "sweetness":          5.2,
                "saltiness":          1.4,
                "sourness":           0.6,
                "bitterness":         0.15,
                "umami":              0.25,
                "astringency":        0.08,
                "globule_density":    0.85,
                "crystal_presence":   0.08,
                "deposit_type":       "clean_ring",
                "predicted_class":    "Fresh",
                "confidence":         0.95,
            },
            # ── AVERAGE: starch addition ──────────────────────────────────────
            {
                "sample_id":          "S002",
                "quality_tier":       "Average",
                "ground_truth":       "Starch",
                "ammonia_signal":     0.45,
                "alcohol_signal":     0.30,
                "voc_signal":         0.50,
                "sulfur_signal":      0.40,
                "hydrocarbon_signal": 0.08,
                "sweetness":          4.5,
                "saltiness":          2.0,
                "sourness":           1.2,
                "bitterness":         0.40,
                "umami":              0.25,
                "astringency":        0.15,
                "globule_density":    0.75,
                "crystal_presence":   0.35,
                "deposit_type":       "dense_deposit",
                "predicted_class":    "Starch",
                "confidence":         0.75,
            },
            # ── WORST: fully spoiled / multi-adulterant ────────────────────────
            {
                "sample_id":          "S003",
                "quality_tier":       "Worst",
                "ground_truth":       "Spoiled",
                "ammonia_signal":     0.95,
                "alcohol_signal":     0.88,
                "voc_signal":         0.97,
                "sulfur_signal":      0.92,
                "hydrocarbon_signal": 0.80,
                "sweetness":          0.8,
                "saltiness":          6.0,
                "sourness":           5.0,
                "bitterness":         4.00,
                "umami":              0.02,
                "astringency":        3.00,
                "globule_density":    0.05,
                "crystal_presence":   0.98,
                "deposit_type":       "crystalline_heavy",
                "predicted_class":    "Spoiled",
                "confidence":         0.99,
            },
        ],
    },
]

# ── Insert batches and samples ────────────────────────────────────────────────
for batch in DEMO_BATCHES:
    batch_id = batch["id"]

    # Insert batch row
    insert_batch(
        batch_id    = batch_id,
        source      = batch["source"],
        description = batch["description"],
        db_path     = DB_PATH,
    )
    print(f"\n[BATCH] {batch_id} — {batch['source']}")

    for s in batch["samples"]:
        sid   = f"{batch_id}-{s['sample_id']}"
        tier  = s["quality_tier"]

        # ── Aroma (E-Nose) ───────────────────────────────────────────────────
        insert_aroma_analysis(
            sample_id = s["sample_id"],
            batch_id       = batch_id,
            ammonia        = s["ammonia_signal"],
            alcohol        = s["alcohol_signal"],
            voc            = s["voc_signal"],
            sulfur         = s["sulfur_signal"],
            hydrocarbon    = s["hydrocarbon_signal"],
            predicted_class= s["predicted_class"],
            confidence     = s["confidence"],
            ground_truth   = s["ground_truth"],
            db_path        = DB_PATH,
        )

        # ── Taste (E-Tongue) ─────────────────────────────────────────────────
        insert_taste_analysis(
            sample_id = s["sample_id"],
            batch_id       = batch_id,
            sweetness      = s["sweetness"],
            saltiness      = s["saltiness"],
            sourness       = s["sourness"],
            bitterness     = s["bitterness"],
            umami          = s["umami"],
            astringency    = s["astringency"],
            predicted_class= s["predicted_class"],
            confidence     = s["confidence"],
            ground_truth   = s["ground_truth"],
            db_path        = DB_PATH,
        )

        # ── Vision ───────────────────────────────────────────────────────────
        obj_det = json.dumps({
            "total_objects_detected": 1,
            "object_class_counts": {s["deposit_type"]: 1},
            "detailed_instances": [{
                "object_class":      s["deposit_type"],
                "confidence":        s["confidence"],
                "bounding_box_xyxy": [10.0, 10.0, 250.0, 250.0],
            }],
        })
        insert_vision_analysis(
            sample_id            = sid,
            batch_id             = batch_id,
            image_path           = f"../data/demo/{tier.lower()}/{s['sample_id'].lower()}.png",
            deposit_type         = s["deposit_type"],
            predicted_class      = s["predicted_class"],
            confidence           = str(s["confidence"]),
            object_detection_info= obj_det,
            ground_truth         = s["ground_truth"],
            db_path              = DB_PATH,
        )

        print(f"  [{s['sample_id']}] [{tier:<7}] gt={s['ground_truth']:<12} "
              f"aroma=({s['ammonia_signal']:.2f}/{s['alcohol_signal']:.2f}/"
              f"{s['voc_signal']:.2f})  "
              f"taste=sweet:{s['sweetness']} bitter:{s['bitterness']}  "
              f"deposit={s['deposit_type']}  conf={s['confidence']}")

print("\n[OK] Demo batches inserted successfully.")

# ── Quick verification ───────────────────────────────────────────────────────
demo_batch_ids = [b["id"] for b in DEMO_BATCHES]
placeholders   = ",".join(f"'{b}'" for b in demo_batch_ids)

for table in ("aroma_analysis", "taste_analysis", "vision_analysis"):
    col    = "batch_id"
    rows   = execute_query(DB_PATH, f"SELECT COUNT(*) as c FROM {table} WHERE {col} IN ({placeholders})")
    count  = dict(rows[0])["c"]
    print(f"  {table}: {count} demo records")